# OpenAI API (gpt-4o-mini) — 4-Condition RACE Experiment

## Mutual Awareness in Semantic Communication: blind / a_aware / b_aware / mutual

Replicates `qwen_agent_ext.ipynb` Key Idea 1 (4-condition extension) using OpenAI API instead of local Qwen3-4B.

- **A (Summarizer)**: Reads passage, writes concise summary under token budget
- **B (Answerer)**: Receives summary + question + choices, outputs answer (never sees passage)
- 2x2 design: A's info level (blind vs choices) x B's awareness (unaware vs "first sentence" hint)
- A caching: blind output reused for blind & b_aware; choices output reused for a_aware & mutual
- Token budgets: 16 / 32 / 48 / 64
- 30 RACE-high questions, seed=42, temperature=0

### Conditions (2x2)

| Condition | A sees | B knows |
|-----------|--------|---------|
| blind | passage only | nothing |
| a_aware | passage + choices | nothing |
| b_aware | passage only | "A puts key info in first sentence" |
| mutual | passage + choices | "A puts key info in first sentence" |

In [ ]:
# ============================================================
# Cell 1: Setup — OpenAI API + RACE loader
# ============================================================
!pip install -q openai datasets

import os
import random
import re
from openai import OpenAI
from datasets import load_dataset

# API Key (set in Colab or environment)
# os.environ["OPENAI_API_KEY"] = "sk-..."  # uncomment and fill
client = OpenAI()

MODEL = "gpt-4o-mini"

def chat(system_prompt, user_message, max_tokens=256):
    """OpenAI API call wrapper."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=0,
    )
    text = response.choices[0].message.content.strip()
    tokens = response.usage.completion_tokens
    return {"response": text, "tokens": tokens}

# RACE loader
print("Loading RACE...")
race = load_dataset("ehovy/race", "high", split="test")

def format_race(ex):
    choices = ex["options"]
    choices_text = "\n".join(f"{chr(65+i)}) {c}" for i, c in enumerate(choices))
    return {
        "passage": ex["article"],
        "question": ex["question"],
        "choices_text": choices_text,
        "answer": ex["answer"],
    }

def sample_race(n, seed=42):
    random.seed(seed)
    samples = random.sample(list(race), min(n, len(race)))
    return [format_race(s) for s in samples]

def extract_answer(response):
    text = response.strip()
    if text.upper() in ("A", "B", "C", "D"):
        return text.upper()
    m = re.search(r'\\boxed\{([A-D])\}', text)
    if m: return m.group(1)
    m = re.search(r'(?:answer|choice)[\s:is]*([A-D])\b', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'^([A-D])[)\.]', text, re.MULTILINE)
    if m: return m.group(1)
    m = re.search(r'\b([A-D])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"RACE loaded: {len(race)} questions")
print(f"Model: {MODEL}")

In [ ]:
# ============================================================
# Cell 2: 4-Condition Experiment (blind / a_aware / b_aware / mutual)
# RACE 30 questions x 4 token budgets x 4 conditions
# A caching: blind/b_aware share A output, a_aware/mutual share A output
# OpenAI API (gpt-4o-mini, temperature=0)
# ============================================================

# --- A prompts ---
A_BLIND = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. Start with the most important fact. "
    "Capture the most important facts and events."
)

A_CHOICES = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader must choose between the answer options shown below. "
    "Start with the ONE fact that best distinguishes between the options. "
    "Do NOT indicate which option is correct."
)

# --- B prompts ---
B_BLIND = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)

B_AWARE = (
    "You are an Answerer. You receive a summary from a Summarizer agent. "
    "IMPORTANT: The Summarizer always puts the most critical information "
    "in the FIRST sentence. Base your answer primarily on the first sentence. "
    "Output ONLY a single letter: A, B, C, or D."
)

# --- 4 conditions (2x2) ---
CONDITIONS = {
    "blind":   {"a_type": "blind",   "b_knows": False},
    "a_aware": {"a_type": "choices", "b_knows": False},
    "b_aware": {"a_type": "blind",   "b_knows": True},
    "mutual":  {"a_type": "choices", "b_knows": True},
}

TX_BUDGETS = [16, 32, 48, 64]

# Sample 30 RACE questions (same seed!)
Q1 = sample_race(30, seed=42)
print(f"Sampled {len(Q1)} RACE questions (seed=42)")
for q in Q1[:3]:
    print(f"  Q: {q['question'][:50]}... ans={q['answer']}")

def run_4cond(questions):
    results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

    for qi, q in enumerate(questions):
        print(f"\n--- Q{qi+1}/{len(questions)}: {q['question'][:60]}...")
        print(f"    Expected: {q['answer']}")

        for budget in TX_BUDGETS:
            # A caching: only 2 API calls per (question, budget)
            # blind output reused for blind & b_aware
            # choices output reused for a_aware & mutual
            a_cache = {}

            a_cache["blind"] = chat(
                A_BLIND,
                f"Passage:\n{q['passage']}",
                max_tokens=budget,
            )

            a_cache["choices"] = chat(
                A_CHOICES,
                f"Passage:\n{q['passage']}\n\nAnswer options:\n{q['choices_text']}",
                max_tokens=budget,
            )

            for cond, cfg in CONDITIONS.items():
                # Reuse cached A output
                a_out = a_cache[cfg["a_type"]]

                # Select B prompt based on awareness
                if cfg["b_knows"]:
                    b_prompt = B_AWARE
                else:
                    b_prompt = B_BLIND

                b_msg = (
                    f"Summary:\n{a_out['response']}\n\n"
                    f"Question: {q['question']}\nChoices:\n{q['choices_text']}\n\nAnswer:"
                )
                b_out = chat(b_prompt, b_msg, max_tokens=48)
                final_ans = extract_answer(b_out["response"])
                correct = final_ans == q["answer"]

                results[budget][cond].append({
                    "correct": correct,
                    "answer": final_ans,
                    "expected": q["answer"],
                    "a_tokens": a_out["tokens"],
                })

        # Log @lowest budget
        b = TX_BUDGETS[0]
        row = "  ".join(
            f"{c}: {results[b][c][-1]['answer']}"
            f"{'✓' if results[b][c][-1]['correct'] else '✗'}"
            for c in CONDITIONS
        )
        print(f"  @{b}tok: {row}")

    return results

print(f"\nRunning 4-condition experiment (gpt-4o-mini, temp=0)...")
print(f"Budgets: {TX_BUDGETS}, Questions: {len(Q1)}, Conditions: {list(CONDITIONS.keys())}")
# A calls: 2 per (question, budget) due to caching; B calls: 4 per (question, budget)
n_a = len(Q1) * len(TX_BUDGETS) * 2
n_b = len(Q1) * len(TX_BUDGETS) * 4
print(f"Total API calls: {n_a} (A, cached) + {n_b} (B) = {n_a + n_b}")
k1_results = run_4cond(Q1)

# ============================================================
# Results table
# ============================================================
print(f"\n{'='*55}")
print(f"  gpt-4o-mini — 4 Conditions (RACE-high, n={len(Q1)})")
print(f"{'='*55}")
print(f"\n{'Budget':<8} {'blind':<10} {'a_aware':<10} {'b_aware':<10} {'mutual':<10}")
print("-" * 48)
for budget in TX_BUDGETS:
    row = f"{budget}tok  "
    for cond in CONDITIONS:
        res = k1_results[budget][cond]
        n = len(res)
        acc = sum(r["correct"] for r in res) / n
        row += f"{acc*100:>5.0f}%    "
    print(row)

# Rate-distortion curve
print(f"\n--- Rate-Distortion Curve ---")
for cond in CONDITIONS:
    points = " -> ".join(
        f"{b}({sum(r['correct'] for r in k1_results[b][cond])/len(k1_results[b][cond])*100:.0f}%)"
        for b in TX_BUDGETS
    )
    print(f"  {cond:<12}: {points}")

# Key findings
print(f"\n--- Key Findings ---")
for budget in TX_BUDGETS:
    accs = {c: sum(r["correct"] for r in k1_results[budget][c]) / len(k1_results[budget][c]) for c in CONDITIONS}
    print(f"  @{budget}tok: blind {accs['blind']*100:.0f}% | a_aware {accs['a_aware']*100:.0f}% | b_aware {accs['b_aware']*100:.0f}% | mutual {accs['mutual']*100:.0f}%")

# Average A tokens actually used (API can stop early)
print(f"\n--- Avg A tokens used (API early stopping) ---")
for budget in TX_BUDGETS:
    for a_type in ["blind", "choices"]:
        # Pick a condition that uses this a_type
        cond = "blind" if a_type == "blind" else "a_aware"
        res = k1_results[budget][cond]
        avg_tok = sum(r["a_tokens"] for r in res) / len(res)
        print(f"  @{budget}tok A_{a_type:<8}: avg {avg_tok:.1f} tokens used")

# 2x2 interaction analysis
print(f"\n--- 2x2 Interaction (A awareness x B awareness) ---")
for budget in TX_BUDGETS:
    accs = {c: sum(r["correct"] for r in k1_results[budget][c]) / len(k1_results[budget][c]) * 100 for c in CONDITIONS}
    a_effect = ((accs["a_aware"] - accs["blind"]) + (accs["mutual"] - accs["b_aware"])) / 2
    b_effect = ((accs["b_aware"] - accs["blind"]) + (accs["mutual"] - accs["a_aware"])) / 2
    interaction = (accs["mutual"] - accs["a_aware"]) - (accs["b_aware"] - accs["blind"])
    print(f"  @{budget}tok: A_effect={a_effect:+.1f}pp  B_effect={b_effect:+.1f}pp  interaction={interaction:+.1f}pp")

print(f"\n--- Compare with Qwen3-4B local results ---")
print("Note: gpt-4o-mini uses proper max_tokens (model can stop early),")
print("while Qwen3-4B local uses greedy decoding (always generates max tokens).")
print("This may lead to different rate-distortion characteristics.")